# Nettoyage des ventes de Seine-et-Marne (DVF 2025)

**Objectif :** partir des 23 529 ventes brutes de Seine-et-Marne et ne garder que des ventes d'un seul logement, vendues au prix de marché, avec une cible calculable (`prix_m2`).

Trois règles de filtrage sont appliquées, une par une, en mesurant à chaque fois ce qu'elles retirent.

In [334]:
import pandas as pd

df = pd.read_parquet("../data/processed/dvf_77_2025.parquet")
print(df.shape)

(64913, 40)


## 1. Une anomalie à corriger avant tout filtrage

Avant d'appliquer les règles, on vérifie qu'une même vente (`id_mutation`) ne porte jamais plusieurs natures de mutation différentes. Si c'était le cas, un filtre sur `nature_mutation` « couperait » la vente en deux : il garderait certaines de ses lignes et en supprimerait d'autres.

In [335]:
natures_par_vente = df.groupby("id_mutation")["nature_mutation"].nunique()
print(natures_par_vente.max())
natures_par_vente[natures_par_vente > 1]

2


id_mutation
2025-1050579    2
Name: nature_mutation, dtype: int64

In [336]:
colonnes = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
            "nom_commune", "type_local", "surface_reelle_bati", "nature_culture"]
df[df["id_mutation"] == "2025-1050579"][colonnes]

,id_mutation,date_mutation,nature_mutation,valeur_fonciere,nom_commune,type_local,surface_reelle_bati,nature_culture
25823,2025-1050579,2025-12-23,Echange,1.0,Pontault Combault,NaN,NaN,sols
25824,2025-1050579,2025-12-23,Echange,1.0,Pontault Combault,NaN,NaN,sols
25825,2025-1050579,2025-12-23,Vente,1.0,Melun,NaN,NaN,NaN
25826,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25827,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25828,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25829,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25830,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25831,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25832,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN


**Observation :** une seule vente, `2025-1050579`, porte deux natures : 2 lignes d'échange à Pontault-Combault et 11 lignes de vente à Melun, le même jour, pour 1 €. Ce sont très probablement deux opérations distinctes regroupées par erreur sous un même identifiant (`id_mutation` est fabriqué par Etalab, non stable).

**Décision :** exclure cette vente avant toute autre règle.

In [337]:
# 1. Identifiants des ventes qui portent plusieurs natures
ids_a_exclure = natures_par_vente[natures_par_vente > 1].index

# 2. DataFrame sans ces ventes
df_coherent = df[~df["id_mutation"].isin(ids_a_exclure)].copy()

# 3. Vérifications
print(df_coherent.groupby("id_mutation")["nature_mutation"].nunique().max())
print(len(df) - len(df_coherent), "lignes retirées")

1
13 lignes retirées


## 2. Règle 2 : ventes classiques uniquement

Le département compte 6 types de mutation. Seules les ventes classiques (`nature_mutation == "Vente"`) reflètent un prix de marché : on écarte les VEFA (achat sur plan), échanges, adjudications et expropriations.

In [338]:
print(df_coherent["nature_mutation"].value_counts())
print(df_coherent.groupby("nature_mutation")["id_mutation"].nunique())

nature_mutation
Vente                                 58716
Vente en l'état futur d'achèvement     5117
Echange                                 691
Vente terrain à bâtir                   187
Adjudication                            175
Expropriation                            14
Name: count, dtype: int64
nature_mutation
Adjudication                             62
Echange                                 406
Expropriation                            14
Vente                                 21316
Vente en l'état futur d'achèvement     1677
Vente terrain à bâtir                    53
Name: id_mutation, dtype: int64


In [339]:
df_vente = df_coherent[df_coherent["nature_mutation"] == "Vente"].copy()
print(df_vente.shape)
print(df_vente["id_mutation"].nunique(), "ventes")

(58716, 40)
21316 ventes


La règle fait passer le département de 23 528 à 21 316 ventes (2 212 retirées, 9,4 %).

## 3. Règle 1 : un seul logement par vente, sans local commercial

Cette règle ne peut pas se juger ligne par ligne : il faut regarder chaque vente dans son ensemble. On marque les lignes qui sont des logements, puis on compte par vente (`groupby`).

In [340]:
df_vente["est_logement"] = df_vente["type_local"].isin(["Maison", "Appartement"])
logements_par_vente = df_vente.groupby("id_mutation")["est_logement"].sum()
logements_par_vente.value_counts().sort_index()

est_logement
0       4892
1      13950
2       2163
3        135
4         84
5         17
6         31
7         11
8          4
9          6
10         1
11         1
12         3
14         1
16         1
18         1
19         1
28         1
36         1
43         1
44         1
46         1
56         1
65         1
73         1
76         1
88         1
97         1
137        1
146        1
258        1
Name: count, dtype: int64

| Logements par vente | Ventes | Part |
| --- | --- | --- |
| 0 (terrains, dépendances seules, locaux commerciaux) | 4 892 | 22,9 % |
| 1 | 13 950 | 65,4 % |
| 2 | 2 163 | 10,1 % |
| 3 ou plus | 311 | 1,5 % |

**Un cas extrême à comprendre avant de filtrer :**

In [341]:
logement_258 = logements_par_vente[logements_par_vente == 258]
vente_258 = df_vente[df_vente["id_mutation"].isin(logement_258.index)]

print(vente_258["valeur_fonciere"].unique())
print(vente_258["nom_commune"].unique())
print(vente_258["type_local"].value_counts(dropna=False))

[579123.36]
<ArrowStringArray>
['Noisiel']
Length: 1, dtype: str
type_local
Appartement                                 258
Dépendance                                  102
Local industriel. commercial ou assimilé      2
NaN                                           1
Name: count, dtype: int64


Une vente du 18 décembre 2025 porte sur **258 logements à Noisiel** (363 lignes, une seule adresse) pour 579 123 €, soit environ **2 244 €/logement** : ce n'est pas un prix de marché, c'est probablement un ensemble immobilier cédé en bloc (bailleur social, collectivité). Cette vente explique aussi pourquoi Noisiel avait 4,5 lignes par vente contre 2,8 pour le département (notebook `02`) : à elle seule, elle représente la moitié des lignes de la commune.

**Décisions pour ce qui accompagne le logement :**

- **Dépendances : gardées.** Le prix les inclut (surface à 0), légère surestimation du prix au m² assumée comme limite.
- **Terrains : gardés**, mais leur surface n'est jamais ajoutée à la surface habitable (ce serait mélanger deux grandeurs différentes).
- **Locaux commerciaux : exclus.** Un bien mixte relève d'un autre marché, et ne représente que 199 ventes sur 13 950 (1,4 %) : un sacrifice négligeable.

In [342]:
logement_1 = logements_par_vente[logements_par_vente == 1]
vente_1 = df_vente[df_vente["id_mutation"].isin(logement_1.index)].copy()

print(vente_1["type_local"].value_counts(dropna=False))

type_local
Dépendance                                  11810
Maison                                       7881
Appartement                                  6069
NaN                                          5086
Local industriel. commercial ou assimilé      268
Name: count, dtype: int64


In [343]:
vente_1["a_local"] = vente_1["type_local"].isin(["Local industriel. commercial ou assimilé"])
local_par_vente = vente_1.groupby("id_mutation")["a_local"].any()
ids_avec_local = local_par_vente[local_par_vente].index

df_regle1 = vente_1[~vente_1["id_mutation"].isin(ids_avec_local)].copy()
print(df_regle1["id_mutation"].nunique(), "ventes")

13751 ventes


In [344]:
# Vérifications : plus de local commercial, et exactement 1 logement par vente
print(df_regle1["type_local"].value_counts(dropna=False))

df_regle1["est_logement"] = df_regle1["type_local"].isin(["Maison", "Appartement"])
verif = df_regle1.groupby("id_mutation")["est_logement"].sum()
print(verif.max(), verif.min())

type_local
Dépendance     11636
Maison          7751
Appartement     6000
NaN             4803
Name: count, dtype: int64
1 1


La règle fait passer le département de 21 316 à 13 751 ventes (7 565 retirées).

## 4. Règle 3 : prix et surface renseignés

Ne garder que les lignes de logement donne une ligne par vente (puisque chaque vente n'en a qu'un), avec le prix et la surface habitable directement accessibles.

In [345]:
df_logements = df_regle1[df_regle1["type_local"].isin(["Maison", "Appartement"])].copy()

print(df_logements["id_mutation"].nunique() == len(df_logements))
print(df_logements["valeur_fonciere"].isna().sum(), "ventes sans prix")
print(df_logements["surface_reelle_bati"].isna().sum(), "ventes sans surface")
print((df_logements["surface_reelle_bati"] == 0.0).sum(), "logements avec une surface de 0")

True
28 ventes sans prix
1 ventes sans surface
0 logements avec une surface de 0


28 ventes sans prix et 1 vente sans surface (29 ventes distinctes). On n'impute jamais la cible : ces ventes sont exclues plutôt que complétées.

In [346]:
ids_sans_prix = df_logements.loc[df_logements["valeur_fonciere"].isna(), "id_mutation"]
ids_sans_surface = df_logements.loc[df_logements["surface_reelle_bati"].isna(), "id_mutation"]
ids_a_exclure_regle3 = pd.concat([ids_sans_prix, ids_sans_surface])

# df_regle123 garde toutes les lignes (logement, dépendances, terrains) des ventes retenues
df_regle123 = df_regle1[~df_regle1["id_mutation"].isin(ids_a_exclure_regle3)].copy()

# df_logements : une ligne par vente, prête pour le calcul du prix au m²
df_logements = df_regle123[df_regle123["type_local"].isin(["Maison", "Appartement"])].copy()

print(df_logements["id_mutation"].nunique() == len(df_logements))
print(df_logements["valeur_fonciere"].isna().sum())
print(df_logements["surface_reelle_bati"].isna().sum())
print(len(df_logements), "ventes")

True
0
0
13722 ventes


**Bilan des règles :** 23 529 → 23 528 (vente incohérente) → 21 316 (règle 2) → 13 751 (règle 1) → **13 722 ventes** (règle 3), soit 58 % des ventes brutes du département.

## 5. Construction des variables dérivées

### Le prix au m², la cible de l'analyse

In [347]:
df_logements["prix_m2"] = df_logements["valeur_fonciere"] / df_logements["surface_reelle_bati"]
df_logements["prix_m2"].describe()

count     13722.000000
mean       3146.983883
std        2797.664085
min           0.012500
25%        2380.952381
50%        3069.760770
75%        3767.248062
max      300000.000000
Name: prix_m2, dtype: float64

La médiane (3 070 €/m²) est réaliste pour la Seine-et-Marne, mais les extrêmes ne le sont pas : 0,01 €/m² (une vente à 1 €) et 300 000 €/m² (une surface de 1 m²). Ces valeurs seront traitées dans le prochain notebook, lors de l'analyse de la distribution des prix (question d'analyse n°1).

### Le nombre de dépendances par vente

Calculé par `groupby` sur `df_regle123` (qui contient encore les lignes de dépendances), puis rattaché à chaque vente avec un `merge`.

In [348]:
df_regle123["est_dependance"] = df_regle123["type_local"].isin(["Dépendance"])

nb_dependances = df_regle123.groupby("id_mutation")["est_dependance"].sum()
nb_dependances = nb_dependances.rename("nb_dependances").reset_index()

df_final = pd.merge(df_logements, nb_dependances, on="id_mutation", how="left")

print(len(df_final))
print(df_final["nb_dependances"].value_counts().sort_index())
print(df_final["nb_dependances"].isna().sum())

13722
nb_dependances
0     5734
1     5050
2     2383
3      458
4       79
5       10
6        3
7        2
10       1
18       1
19       1
Name: count, dtype: int64
0


42 % des ventes n'ont aucune dépendance, 37 % en ont une. Cas suspects (18 et 19 dépendances pour un seul logement) à noter pour l'analyse des valeurs extrêmes.

### La surface de terrain par vente

**Piège à éviter :** une même parcelle peut apparaître sur plusieurs lignes d'une vente (une par local sur cette parcelle). L'additionner directement la compterait plusieurs fois.

In [349]:
colonnes = ["id_mutation", "type_local", "surface_reelle_bati",
            "id_parcelle", "nature_culture", "surface_terrain"]

# Exemple : une maison avec dépendance(s) et terrain
conditions = (df_final["nb_dependances"] > 0) & (df_final["type_local"] == "Maison") & (df_final["surface_terrain"] > 0)
exemple = df_final.loc[conditions, "id_mutation"].iloc[0]

df_regle123.loc[df_regle123["id_mutation"] == exemple, colonnes]

,id_mutation,type_local,surface_reelle_bati,id_parcelle,nature_culture,surface_terrain
23,2025-1040923,Maison,83.0,77296000AC0478,sols,104.0
24,2025-1040923,Dépendance,0.0,77296000AC0478,sols,104.0
25,2025-1040923,NaN,NaN,77296000AC0479,sols,17.0
26,2025-1040923,NaN,NaN,77296000AC0832,sols,68.0


La même parcelle (même `id_parcelle`) apparaît bien sur la ligne « Maison » et sur la ligne « Dépendance ». Il faut donc supprimer les doublons de parcelle **avant** d'additionner les surfaces de terrain.

In [350]:
terrains = df_regle123[["id_mutation", "id_parcelle", "nature_culture", "surface_terrain"]].drop_duplicates()

terrain_par_vente = terrains.groupby("id_mutation")["surface_terrain"].sum()
terrain_par_vente = terrain_par_vente.rename("surface_terrain_total").reset_index()

df_final = pd.merge(df_final, terrain_par_vente, on="id_mutation", how="left")
print(len(df_final))

df_final.loc[df_final["id_mutation"] == exemple, ["id_mutation", "surface_terrain_total"]].drop_duplicates()

13722


,id_mutation,surface_terrain_total
11,2025-1040923,189.0


Pour l'exemple ci-dessus, la surface totale correcte est 189 m² (104 + 17 + 68), sans compter deux fois la parcelle partagée par la maison et sa dépendance.

Pour les appartements, qui n'ont en général pas de terrain en propre :

In [351]:
apparts = df_final[df_final["type_local"] == "Appartement"]
print(apparts["surface_terrain_total"].isna().sum(), "valeurs manquantes")
print((apparts["surface_terrain_total"] == 0).sum(), "appartements avec 0 m² de terrain, sur", len(apparts))

0 valeurs manquantes
5760 appartements avec 0 m² de terrain, sur 5991


Aucune valeur manquante : un appartement sans terrain obtient 0 m² (la somme d'aucune valeur vaut 0 avec `.sum()`), ce qui correspond bien à la réalité. Environ 4 % des appartements ont tout de même du terrain (rez-de-chaussée avec jardin privatif, par exemple).

## 6. Jeu de données final

Sélection des colonnes utiles aux 5 questions d'analyse, regroupées par thème.

In [352]:
colonnes_finales = [
    # Identification de la vente
    "id_mutation",
    "date_mutation",

    # Prix
    "valeur_fonciere",
    "prix_m2",

    # Caractéristiques du bien
    "type_local",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "nb_dependances",
    "surface_terrain_total",

    # Localisation
    "code_commune",
    "nom_commune",
    "code_postal",
    "longitude",
    "latitude",
]

df_propre = df_final[colonnes_finales]
df_propre.to_parquet("../data/processed/dvf_77_2025_propre.parquet", index=False)

In [353]:
df_verif = pd.read_parquet("../data/processed/dvf_77_2025_propre.parquet")
print(df_verif.shape)
df_verif.dtypes

(13722, 14)


id_mutation                      str
date_mutation                    str
valeur_fonciere              float64
prix_m2                      float64
type_local                       str
surface_reelle_bati          float64
nombre_pieces_principales    float64
nb_dependances                 int64
surface_terrain_total        float64
code_commune                     str
nom_commune                      str
code_postal                      str
longitude                    float64
latitude                     float64
dtype: object

## Conclusion

Ce notebook part des 23 529 ventes brutes de Seine-et-Marne (`dvf_77_2025.parquet`) et applique trois règles de filtrage pour ne garder que des ventes d'un seul logement, au prix de marché, avec une cible calculable.

**Règles appliquées :**

1. **Exclusion d'une vente incohérente** (`2025-1050579`) : une même mutation regroupait par erreur un échange et une vente, dans deux communes différentes, au prix symbolique de 1 €.
2. **Ventes classiques uniquement** (`nature_mutation == "Vente"`) : écarte VEFA, échanges, adjudications, expropriations.
3. **Un seul logement par vente, sans local commercial** : découverte au passage d'une vente de 258 logements à Noisiel (579 123 € au total, soit 2 244 €/logement — pas un prix de marché), qui explique aussi pourquoi Noisiel avait 4,5 lignes par vente contre 2,8 pour le département (notebook 02). Les dépendances et les terrains sont gardés ; les 199 ventes avec un local commercial (1,4 % des ventes à un logement) sont exclues.
4. **Prix et surface renseignés** : 29 ventes sans prix ou sans surface sont retirées, sans jamais imputer la cible.

**Variables créées :**

- `prix_m2` : la cible de l'analyse (prix / surface habitable).
- `nb_dependances` : nombre de dépendances par vente (`groupby` + `merge`).
- `surface_terrain_total` : surface de terrain par vente, en évitant un piège découvert en chemin — une même parcelle apparaît sur plusieurs lignes d'une vente (une par local), et l'additionner sans précaution la comptait plusieurs fois (`drop_duplicates()` avant la somme).

**Bilan :** de 23 529 à **13 722 ventes** (58 %), enregistrées dans `data/processed/dvf_77_2025_propre.parquet` avec 14 colonnes utiles à l'analyse.

**À traiter dans le prochain notebook :** le `describe()` de `prix_m2` montre des valeurs extrêmes (0,01 €/m², 300 000 €/m²), pas encore analysées. C'est la question d'analyse n°1.